In [1]:
import os
import glob
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

In [2]:
# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 16:41:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# snapshot date???

# get labels

In [4]:
print(f'\n{"="*60}')
print('MODEL TRAINING')
print(f'{"="*60}')


MODEL TRAINING


In [5]:
print('\n--- Join features and labels ---')
# connect to label store
label_folder_path = "datamart/gold/label_store/gold_label_store_*"
label_store_sdf = spark.read.parquet(label_folder_path)
print(f'Loaded from: {label_folder_path}')
print("row_count:",label_store_sdf.count())

# rename snapshot_date col to indicate mob6
label_store_sdf = label_store_sdf.withColumnRenamed('snapshot_date', 'mob6_snapshot_date')




--- Join features and labels ---


Loaded from: datamart/gold/label_store/gold_label_store_*


[Stage 2:====>                                                    (1 + 11) / 12]

row_count: 12500


In [6]:
# label_store_sdf.printSchema()

# get features

In [7]:
feature_folder_path = "datamart/gold/feature_store/gold_joined.parquet"
feature_store_sdf = spark.read.parquet(feature_folder_path)
print(f'Loaded from: {feature_folder_path}')
print("row_count:",feature_store_sdf.count())

# feature_store_sdf.show(2, truncate=False, vertical = True)

Loaded from: datamart/gold/feature_store/gold_joined.parquet
row_count: 12500


# join feat and label

In [8]:
xy_sdf = feature_store_sdf.join(label_store_sdf, 
                                on='Customer_ID', 
                                how='outer')
print('Feature store joined with label store')

Feature store joined with label store


In [9]:
# xy_sdf.filter(col("snapshot_date").isNull()).limit(1).count() > 0


In [10]:
xyclean_sdf = xy_sdf.drop('label_def', 'loan_id', 'mob6_snapshot_date')
print("Columns dropped in joined table: 'label_def', 'loan_id', 'mob6_snapshot_date'")

Columns dropped in joined table: 'label_def', 'loan_id', 'mob6_snapshot_date'


In [11]:
# xyclean_sdf.show(1, vertical = True)

In [12]:
# xypdf = xyclean_sdf.toPandas()

In [13]:
# xypdf['snapshot_date'].value_counts().sort_index()

# set up config

In [14]:
print('\n--- Split train, test, oot ---')


--- Split train, test, oot ---


In [15]:
# set up config
model_train_date_str = "2024-09-01" # remaining snapshot months: 2024-09-01 to 2025-01-01 (inference?)
train_test_period_months = 15
oot_period_months = 5
train_test_ratio = 0.8

config = {}
config["model_train_date_str"] = model_train_date_str
config["train_test_period_months"] = train_test_period_months
config["oot_period_months"] =  oot_period_months
config["model_train_date"] =  datetime.strptime(model_train_date_str, "%Y-%m-%d")
config["oot_end_date"] =  config['model_train_date'] - timedelta(days = 1)
config["oot_start_date"] =  config['model_train_date'] - relativedelta(months = oot_period_months)
config["train_test_end_date"] =  config["oot_start_date"] - timedelta(days = 1)
config["train_test_start_date"] =  config["oot_start_date"] - relativedelta(months = train_test_period_months)
config["train_test_ratio"] = train_test_ratio 

print("Split configurations:")
pprint.pprint(config)

Split configurations:
{'model_train_date': datetime.datetime(2024, 9, 1, 0, 0),
 'model_train_date_str': '2024-09-01',
 'oot_end_date': datetime.datetime(2024, 8, 31, 0, 0),
 'oot_period_months': 5,
 'oot_start_date': datetime.datetime(2024, 4, 1, 0, 0),
 'train_test_end_date': datetime.datetime(2024, 3, 31, 0, 0),
 'train_test_period_months': 15,
 'train_test_ratio': 0.8,
 'train_test_start_date': datetime.datetime(2023, 1, 1, 0, 0)}


In [16]:
oot_sdf = xyclean_sdf.filter(
    (col('snapshot_date')>= config['oot_start_date']) &
    (col('snapshot_date')<= config['oot_end_date'])
)

oot_pdf = oot_sdf.toPandas()
print("OOT row counts:")
print(oot_pdf['snapshot_date'].value_counts().sort_index())

OOT row counts:
snapshot_date
2024-04-01    513
2024-05-01    491
2024-06-01    498
2024-07-01    505
2024-08-01    543
Name: count, dtype: int64


In [17]:
traintest_sdf = xyclean_sdf.filter(
    (col('snapshot_date') <= config['train_test_end_date']) &
    (col('snapshot_date') >= config['train_test_start_date'])
)

traintest_pdf = traintest_sdf.toPandas()


In [18]:
# convert snapshot_date to year and month columns
traintest_pdf["snapshot_date"] = pd.to_datetime(traintest_pdf["snapshot_date"])
traintest_pdf["snapshot_year"]  = traintest_pdf["snapshot_date"].dt.year
traintest_pdf["snapshot_month"] = traintest_pdf["snapshot_date"].dt.month
traintest_pdf["snapshot_quarter"] = traintest_pdf["snapshot_date"].dt.quarter

oot_pdf["snapshot_date"] = pd.to_datetime(oot_pdf["snapshot_date"])
oot_pdf["snapshot_year"]  = oot_pdf["snapshot_date"].dt.year
oot_pdf["snapshot_month"] = oot_pdf["snapshot_date"].dt.month
oot_pdf["snapshot_quarter"] = oot_pdf["snapshot_date"].dt.quarter

# define feature and label columns
exclude_cols = ['Customer_ID', 'label', 'snapshot_date']
feature_cols = [c for c in traintest_pdf.columns if c not in exclude_cols]
print(f'\nFeature columns ({len(feature_cols)} total): {feature_cols}')



Feature columns (60 total): ['Age', 'Occupation', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Payment_of_Min_Amount', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance', 'Credit_History_Age_Years', 'Loan_Auto_Loan', 'Loan_Credit_Builder_Loan', 'Loan_Debt_Consolidation_Loan', 'Loan_Home_Equity_Loan', 'Loan_Mortgage_Loan', 'Loan_Not_Specified', 'Loan_Payday_Loan', 'Loan_Personal_Loan', 'Loan_Student_Loan', 'Spending_Behaviour', 'Payments_Size', 'months12_to_annual_income_ratio', 'EMI_to_Salary_Ratio', 'Debt_to_Income', 'Disposable_Income', 'fe_1_mean', 'fe_2_mean', 'fe_3_mean', 'fe_4_mean', 'fe_5_mean', 'fe_6_mean', 'fe_7_mean', 'fe_8_mean', 'fe_9_mean', 'fe_10_mean', 'fe_11_mean', 'fe_12_mean', 'fe_13_mean', 'fe_14_mean', 'fe_15_mean', 'fe_16_

In [19]:
X_temp = traintest_pdf[feature_cols].copy()
y_temp = traintest_pdf['label'].copy()

X_oot = oot_pdf[feature_cols].copy()
y_oot = oot_pdf['label'].copy()

# train/test split — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.2,
    random_state=55,
    stratify=y_temp
)


sets = [
    ("train", y_train),
    ("test",  y_test),
    ("oot",   y_oot),
]
for name, y in sets:
    print(f"{name:6} | rows: {len(y):>7,} | bad rate: {y.mean():.4f}")

train  | rows:   5,977 | bad rate: 0.2871
test   | rows:   1,495 | bad rate: 0.2870
oot    | rows:   2,550 | bad rate: 0.2996


### EDA

### Continue model train

In [ ]:
print('\n--- Post-split processing ---')

In [ ]:
# mean imputation
mean_cols = ['Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan', 
             'Num_of_Delayed_Payment', 'Age']
mean_impute_vals = {}
for c in mean_cols:
    mean_val = X_train[c].mean()
    mean_impute_vals[c] = mean_val
    X_train[c] = X_train[c].fillna(mean_val)
    X_test[c]  = X_test[c].fillna(mean_val)        
    X_oot[c]   = X_oot[c].fillna(mean_val)
print(f'Mean imputation applied to: {mean_cols}')

In [ ]:
# median imputation
median_cols = ['Delay_from_due_date', 'Num_Credit_Inquiries', 
               'Monthly_Balance', 'Interest_Rate']
median_impute_vals = {}
for c in median_cols:
    median_val = X_train[c].median()
    median_impute_vals[c] = median_val
    X_train[c] = X_train[c].fillna(median_val)
    X_test[c]  = X_test[c].fillna(median_val)      
    X_oot[c]   = X_oot[c].fillna(median_val)       
print(f'Median imputation applied to: {median_cols}')

In [ ]:
# clickstream fe_ imputation — fit on train clickers only
fe_cols = [f'fe_{i}_mean' for i in range(1, 21)]
fe_impute_vals = {}
for c in fe_cols:
    mean_val = X_train.loc[X_train['has_clickstream'] == 1, c].mean() 
    fe_impute_vals[c] = mean_val
    X_train[c] = X_train[c].fillna(mean_val)
    X_test[c]  = X_test[c].fillna(mean_val)
    X_oot[c]   = X_oot[c].fillna(mean_val)
print(f'Clickstream mean imputation applied to: {fe_cols}')

In [ ]:
    # # 3. post-split processing
    # print('\n--- Post-split processing ---')

    # # mean imputation — fit on train only
    # mean_cols = ['Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan', 
    #              'Num_of_Delayed_Payment', 'Age']
    # for c in mean_cols:
    #     mean_val = X_train[c].mean()                   
    #     X_train[c] = X_train[c].fillna(mean_val)
    #     X_test[c]  = X_test[c].fillna(mean_val)        
    #     X_oot[c]   = X_oot[c].fillna(mean_val)
    # print(f'Mean imputation applied to: {mean_cols}')

    # # median imputation — fit on train only
    # median_cols = ['Delay_from_due_date', 'Num_Credit_Inquiries', 
    #                'Monthly_Balance', 'Interest_Rate']
    # for c in median_cols:
    #     median_val = X_train[c].median()               
    #     X_train[c] = X_train[c].fillna(median_val)
    #     X_test[c]  = X_test[c].fillna(median_val)      
    #     X_oot[c]   = X_oot[c].fillna(median_val)       
    # print(f'Median imputation applied to: {median_cols}')

    
    # # clickstream fe_ imputation — fit on train clickers only
    # fe_cols = [f'fe_{i}_mean' for i in range(1, 21)]
    # for c in fe_cols:
    #     mean_val = X_train.loc[X_train['has_clickstream'] == 1, c].mean() 
    #     X_train[c] = X_train[c].fillna(mean_val)
    #     X_test[c]  = X_test[c].fillna(mean_val)
    #     X_oot[c]   = X_oot[c].fillna(mean_val)
    # print(f'Clickstream mean imputation applied to: {fe_cols}')

    # dummy encoding for object columns
    cat_cols = ['Occupation', 'Credit_Mix', 'Payment_of_Min_Amount', 
                'Spending_Behaviour', 'Payments_Size']
    X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
    X_test  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=True)
    X_oot   = pd.get_dummies(X_oot,   columns=cat_cols, drop_first=True)
    print(f'Dummy encoding applied to: {cat_cols}')
    print(f'Feature columns after encoding ({len(X_train.columns)} total): {list(X_train.columns)}')
    
    # scaling for 
    scaler = StandardScaler()
    scale_cols = [
        # continuous numeric — large magnitude differences
        'Age', 'Annual_Income', 'Monthly_Inhand_Salary',
        'Outstanding_Debt', 'Total_EMI_per_month',
        'Amount_invested_monthly', 'Monthly_Balance',
        'Changed_Credit_Limit', 'Credit_Utilization_Ratio',
        'Credit_History_Age_Years', 'snapshot_year', 'snapshot_month',
        
        # count columns — different ranges
        'Num_Bank_Accounts', 'Num_Credit_Card', 'Num_of_Loan',
        'Num_Credit_Inquiries', 'Num_of_Delayed_Payment',
        'Delay_from_due_date', 'Interest_Rate', 'clickstream_days',
        
        # engineered features
        'EMI_to_Salary_Ratio', 'Debt_to_Income', 'Disposable_Income',
        'months12_to_annual_income_ratio',

        # loan counts
        'Loan_Auto_Loan', 'Loan_Credit_Builder_Loan',
        'Loan_Debt_Consolidation_Loan', 'Loan_Home_Equity_Loan',
        'Loan_Mortgage_Loan', 'Loan_Not_Specified', 'Loan_Payday_Loan',
        'Loan_Personal_Loan', 'Loan_Student_Loan'
    ] + [f'fe_{i}_mean' for i in range(1, 21)]

    print(f'Standard scaling applied to {len(scale_cols)} columns')
    
    X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
    X_test[scale_cols]  = scaler.transform(X_test[scale_cols])
    X_oot[scale_cols]   = scaler.transform(X_oot[scale_cols])

In [20]:
# SAVE INFERENCE SET SEPARATELY?